# LAB022-024 fermentation profiles

Each column represents one lab. Each row shares a vertical scale across labs. Time is measured from inoculation using the local sampling schedule in the offline workbook.

The Y15 labels `LAB02x-1` and `LAB02x-2` are excluded. `LAB02x-3` supplies sample 01 at inoculation (0 h), and samples 02 through 06 keep their numbering. The unnumbered post-nutrition sample was taken at 11:30 on 9 November 2026 and is marked with a diamond on the YAN row.

In [ ]:
import csv
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd

%matplotlib inline

cwd = Path.cwd().resolve()
data_dir = next(
    path / 'data/original/natural/lab022_024'
    for path in (cwd, *cwd.parents)
    if (path / 'data/original/natural/lab022_024').is_dir()
)
labs = ['LAB022', 'LAB023', 'LAB024']
workbook = data_dir / 'lab022-024MedicionesOffline_estandar.xlsx'
experiments = pd.read_excel(workbook, sheet_name='Experimentos').set_index('CodigoLab').loc[labs]
inoculation = experiments['FechaHora_Inoculacion']

offline = (
    pd.read_excel(workbook, sheet_name='MedicionesOffline')
    .rename(columns={'CodigoLab': 'lab', 'Muestra': 'sample'})
    .loc[lambda frame: frame['lab'].isin(labs)]
    .copy()
)
offline['sampled_at'] = [
    datetime.combine(date.date(), time)
    for date, time in zip(offline['Fecha'], offline['Hora'])
]
nutrition = (
    offline.loc[offline['Observaciones'].eq('Hora de nutrición')]
    .set_index('lab')['sampled_at']
    .loc[labs]
)
offline = offline.dropna(subset='sample').copy()
offline['sample'] = offline['sample'].replace(
    {**{number: f'{number:02d}' for number in range(1, 7)}, 'POST NUT': 'POSTNUT'}
)
offline = offline.sort_values(['lab', 'sampled_at'])

def elapsed_hours(frame):
    return (frame['sampled_at'] - frame['lab'].map(inoculation)).dt.total_seconds() / 3600

offline['hours'] = elapsed_hours(offline)
sample_times = offline[['lab', 'sample', 'sampled_at']]
post_times = sample_times.loc[sample_times['sample'].eq('POSTNUT')].set_index('lab')['sampled_at']
assert post_times.eq(pd.Timestamp('2026-11-09 11:30')).all()
assert post_times.gt(nutrition).all()


## Y15 sample selection

Samples 01–04 use the `EXPAuto` export, which gives a single value per assay after exact duplicates are removed. The 2 November `ONLINE` export contains overlapping records and alternate values for some of these assays. Samples 05, 06, and the 9 November `POSTNUT` use the 9 November `ONLINE` export. An explicit label map keeps `-3` (initial condition) distinct from `-03` (sample 03) and excludes `-1` and `-2`.

In [ ]:
y15_sources = {
    'EXPAuto(2026-11-02 11-57).txt': {'3': '01', '02': '02', '03': '03', '04': '04'},
    'ONLINE(26-11-09 09-52)_1.txt': {'05': '05', '06': '06', 'POSTNUT': 'POSTNUT'},
}
y15_columns = ['source_label', 'assay', 'station', 'reading', 'unit', 'assayed_at']
assays = ['PAN', 'AMMONIA', 'YAN', 'GLUCOSE-320', 'GLU-FRU-320']
y15_frames = []
for filename, samples in y15_sources.items():
    labels = {
        f'{lab}-{suffix}': (lab, sample)
        for lab in labs
        for suffix, sample in samples.items()
    }
    raw = pd.read_csv(
        data_dir / filename, sep='\t', header=None, names=y15_columns,
        dtype={'reading': str}, encoding='utf-8-sig',
    )
    keep = raw['source_label'].isin(labels) & raw['assay'].isin(assays)
    selected = raw.loc[keep].drop_duplicates().copy()
    selected[['lab', 'sample']] = selected['source_label'].map(labels).tolist()
    selected['assayed_at'] = pd.to_datetime(selected['assayed_at'], format='%d-%m-%Y %H:%M:%S')
    selected['reading'] = pd.to_numeric(selected['reading'].str.replace(',', '.', regex=False))
    selected['source_file'] = filename
    y15_frames.append(selected)

y15 = pd.concat(y15_frames, ignore_index=True).merge(
    sample_times, on=['lab', 'sample'], how='left', validate='many_to_one'
)
assert y15['sampled_at'].notna().all()
assert y15.loc[y15['sample'].eq('01'), 'sampled_at'].eq(
    y15.loc[y15['sample'].eq('01'), 'lab'].map(inoculation)
).all()
assert y15.loc[y15['sample'].eq('POSTNUT'), 'assayed_at'].dt.date.eq(
    datetime(2026, 11, 9).date()
).all()

# Pivot also checks that each lab, sample, and assay has one value.
chemistry = (
    y15.pivot(index=['lab', 'sample', 'sampled_at'], columns='assay', values='reading')
    .reset_index()
    .sort_values(['lab', 'sampled_at'])
)
chemistry['hours'] = elapsed_hours(chemistry)
chemistry['Fructose (derived)'] = chemistry['GLU-FRU-320'] - chemistry['GLUCOSE-320']

# Prefer instrument-reported YAN where available and calculate the remaining values.
ammonia_n_fraction = 0.8225
chemistry['YAN (calculated)'] = chemistry['PAN'] + ammonia_n_fraction * chemistry['AMMONIA']
chemistry['YAN for plot'] = chemistry['YAN'].fillna(chemistry['YAN (calculated)'])

expected_samples = {'01', '02', '03', '04', '05', '06', 'POSTNUT'}
assert chemistry.groupby('lab')['sample'].agg(set).eq(expected_samples).all()
assert chemistry['YAN for plot'].notna().all()
assert chemistry.loc[chemistry['sample'].eq('POSTNUT'), 'YAN'].notna().all()


In [ ]:
oculyze_frames = []
for lab in labs:
    filename = data_dir / f'oculyze_{lab.lower()}.csv'
    # Each CSV record is wrapped in a single quoted field.
    with filename.open(encoding='utf-8-sig', newline='') as file:
        records = [row[0] for row in csv.reader(file) if row]
    measurements = pd.DataFrame(csv.DictReader(records))
    labels = measurements['Name']
    if lab == 'LAB023':
        labels = labels.replace({'LAB024-05': 'LAB023-05'})
    measurements[['lab', 'sample']] = labels.str.rsplit('-', n=1, expand=True)
    assert measurements['lab'].eq(lab).all()
    measurements['sampled_at'] = pd.to_datetime(
        measurements['Description'].str.strip(), format='%d/%m/%y %H:%M'
    )
    for column in ['Concentration', 'Viability', 'Density']:
        measurements[column] = pd.to_numeric(measurements[column])
    oculyze_frames.append(measurements)

oculyze = pd.concat(oculyze_frames, ignore_index=True).sort_values(['lab', 'sampled_at'])
assert oculyze.groupby('lab')['sample'].agg(set).eq(expected_samples - {'POSTNUT'}).all()
density_check = oculyze.merge(
    offline[['lab', 'sample', 'Densidad_g_L']], on=['lab', 'sample'], validate='one_to_one'
)
numbered_later = density_check.loc[density_check['sample'].ne('01')]
assert numbered_later['Density'].eq(numbered_later['Densidad_g_L']).all()
oculyze['hours'] = elapsed_hours(oculyze)
oculyze['Viable concentration'] = oculyze['Concentration'] * oculyze['Viability'] / 100

YAN in mg N/L uses the instrument reading when available and otherwise uses PAN + 0.8225 × ammonia. The factor converts ammonia from mg NH₃/L to mg N/L. The calculation uses the exported ammonia values, including the −2 mg/L reading for LAB024 sample 05. [Australian Wine Research Institute](https://www.awri.com.au/industry_support/winemaking_resources/yan/)

Glucose and glucose + fructose come from the Y15 assays. Fructose is the difference between glucose + fructose and glucose. The glucose + fructose panels also show the historical linear estimate `2.5 × (density − 1000)`, using the offline density in g/L.

Oculyze total cell concentration comes from the CSV exports. Viable cell concentration is total concentration × viability fraction. Sampling times come from the Oculyze descriptions. The row labeled `LAB024-05` in the LAB023 export is assigned to LAB023 because its density (1008.2 g/L) matches LAB023 sample 05 in the offline workbook.

In [ ]:
panels = [
    (chemistry, 'YAN for plot', 'YAN (mg N/L)'),
    (chemistry, 'GLUCOSE-320', 'Glucose (g/L)'),
    (chemistry, 'Fructose (derived)', 'Fructose (g/L)'),
    (chemistry, 'GLU-FRU-320', 'Glucose + fructose (g/L)'),
    (oculyze, 'Viable concentration', 'Oculyze (million cells/mL)'),
]
colors = ['#1767A0', '#BD581C', '#137D60']
nutrition_hours = (nutrition - inoculation).dt.total_seconds() / 3600
density = offline.dropna(subset='Densidad_g_L').copy()
density['Glucose + fructose (estimated)'] = 2.5 * (density['Densidad_g_L'] - 1000)
density_style = dict(color='#555B62', marker='s', markerfacecolor='white',
                     linestyle='--', linewidth=1.3, markersize=4.5)

fig, axes = plt.subplots(
    len(panels), len(labs), figsize=(12, 11.5), dpi=125,
    sharex=True, sharey='row', layout='constrained',
)

for row, (frame, column, label) in enumerate(panels):
    axes[row, 0].set_ylabel(label)
    for col, (lab, color) in enumerate(zip(labs, colors)):
        ax = axes[row, col]
        data = frame.loc[frame['lab'].eq(lab)].dropna(subset=column)
        marker = '^' if frame is oculyze else 'o'
        ax.plot(data['hours'], data[column], color=color, marker=marker,
                markersize=4.5, linewidth=1.6, zorder=3)
        if row == 0:
            setpoint = experiments.at[lab, 'Temperatura_SP_C']
            ax.set_title(f'{lab} | {setpoint:g} °C setpoint', fontweight='bold')
            ax.axvline(nutrition_hours[lab], color='#646D77', linestyle='--', linewidth=1.1)
            post = data.loc[data['sample'].eq('POSTNUT')]
            ax.scatter(post['hours'], post[column], color=color, marker='D', s=58,
                       edgecolor='white', linewidth=0.8, zorder=4)
        if column == 'GLU-FRU-320':
            estimate = density.loc[density['lab'].eq(lab)]
            ax.plot(estimate['hours'], estimate['Glucose + fructose (estimated)'],
                    **density_style, zorder=4)
        if frame is oculyze:
            ax.plot(data['hours'], data['Concentration'], color=color, marker='^',
                    markerfacecolor='white', linestyle=':', linewidth=1.4,
                    markersize=5, zorder=4)
            ax.set_xlabel('Hours since inoculation')
        ax.grid(axis='y', alpha=0.25)
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)

axes[-1, 0].set_xticks(range(0, 121, 20))
fig.suptitle('LAB022-024 fermentation profiles', fontweight='bold', fontsize=15)
fig.legend(
    handles=[
        Line2D([], [], color='#394756', marker='o', label='Y15 measurement'),
        Line2D([], [], **density_style, label='Density-based estimate'),
        Line2D([], [], color='#646D77', linestyle='--', label='Nutrient addition'),
        Line2D([], [], color='#394756', marker='D', linestyle='None',
               label='Post-nutrition sample'),
        Line2D([], [], color='#394756', marker='^', label='Viable cells'),
        Line2D([], [], color='#394756', marker='^', markerfacecolor='white',
               linestyle=':', label='Total cells'),
    ],
    loc='outside lower center', ncols=3, frameon=False,
)
plt.show()